In [1]:
print("Here begins the hybrid models notebook.")

Here begins the hybrid models notebook.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.feature_selection import SelectFromModel

from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)
df_ml = pd.read_csv("../../data/processed/world_happiness_ml.csv")

df_ml.head()

,Country,Year,Happiness_Score,GDP_per_Capita,Social_Support,Healthy_Life_Expectancy,Freedom,Generosity,Corruption_Perception,Unemployment_Rate,Education_Index,Population,Urbanization_Rate,Life_Satisfaction,Public_Trust,Mental_Health_Index,Income_Inequality,Public_Health_Expenditure,Climate_Index,Work_Life_Balance,Internet_Access,Crime_Rate,Political_Stability,Employment_Rate,Year_bin
0,China,2022,4.39,44984.68,0.53,71.11,0.41,-0.05,0.83,14.98,0.52,1311940760,78.71,8.88,0.34,76.44,46.06,8.92,62.75,8.59,74.40,70.30,0.29,61.38,2020
1,UK,2015,5.49,30814.59,0.93,63.14,0.89,0.04,0.84,19.46,0.83,1194240877,50.87,5.03,0.72,53.38,46.43,4.43,53.11,8.76,91.74,73.32,0.76,80.18,2015
2,Brazil,2009,4.65,39214.84,0.03,62.36,0.01,0.16,0.59,16.68,0.95,731100898,48.75,5.22,0.23,82.40,31.03,3.78,33.30,6.06,71.80,28.99,0.94,72.65,2005
3,France,2019,5.20,30655.75,0.77,78.94,0.98,0.25,0.63,2.64,0.70,1293957314,81.78,5.69,0.68,46.87,57.65,4.43,90.59,6.36,86.16,45.76,0.48,55.14,2015
4,China,2022,7.28,30016.87,0.05,50.33,0.62,0.18,0.92,7.70,0.92,1432971455,82.39,6.33,0.50,60.38,28.54,7.66,59.33,3.00,71.10,65.67,0.12,51.55,2020


In [2]:
# Inspect data structure
print(df_ml.shape)
print(df_ml.dtypes)

(4000, 25)
Country                          str
Year                           int64
Happiness_Score              float64
GDP_per_Capita               float64
Social_Support               float64
Healthy_Life_Expectancy      float64
Freedom                      float64
Generosity                   float64
Corruption_Perception        float64
Unemployment_Rate            float64
Education_Index              float64
Population                     int64
Urbanization_Rate            float64
Life_Satisfaction            float64
Public_Trust                 float64
Mental_Health_Index          float64
Income_Inequality            float64
Public_Health_Expenditure    float64
Climate_Index                float64
Work_Life_Balance            float64
Internet_Access              float64
Crime_Rate                   float64
Political_Stability          float64
Employment_Rate              float64
Year_bin                       int64
dtype: object


In [3]:
# Define predictors and target
# - Life_Satisfaction is the regression target
# - Country is excluded because it is a high-cardinality identifier
# - Year_bin is excluded because it duplicates the information in Year
X = df_ml.drop(columns=["Life_Satisfaction", "Country", "Year_bin", "Happiness_Score"])
y = df_ml["Life_Satisfaction"]

# Check shapes
print(X.shape, y.shape)

# Check missing values in predictors and target
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

(4000, 21) (4000,)
Missing values in X: 0
Missing values in y: 0


In [4]:
# Train/test split
# Since this is a regression task, stratification is not used
# The split is still useful for final model comparison, even though cross-validation
# is especially important for this small dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (3200, 21)
Test shape: (800, 21)


In [5]:
# cross-validation baseline for Linear Regression
# With only around 200 rows, cross-validation gives a more stable estimate
# than relying only on a single train/test split

cv = KFold(n_splits=5, shuffle=True, random_state=42)

linreg_cv = LinearRegression()

cv_results_lin = cross_validate(
    linreg_cv,
    X, y,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

print("Linear Regression CV results")
print("Mean MAE:", -cv_results_lin["test_mae"].mean())
print("Mean RMSE:", -cv_results_lin["test_rmse"].mean())
print("Mean R2:", cv_results_lin["test_r2"].mean())

Linear Regression CV results
Mean MAE: 1.2398113581003756
Mean RMSE: 1.4357938826033283
Mean R2: -0.003112198132928956


In [6]:
# Hybrid 1: Random Forest feature selection
# Random Forest is used here as a nonlinear feature selector
# The final model will still be Linear Regression for interpretability

rf_selector = RandomForestRegressor(
    n_estimators=150,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_selector.fit(X_train, y_train)

selector = SelectFromModel(
    rf_selector,
    threshold="median"
)

selector.fit(X_train, y_train)

X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()]

print("Number of selected features:", len(selected_features))
print("\nSelected features:")
print(selected_features.tolist())

Number of selected features: 11

Selected features:
['GDP_per_Capita', 'Generosity', 'Unemployment_Rate', 'Population', 'Urbanization_Rate', 'Mental_Health_Index', 'Income_Inequality', 'Climate_Index', 'Work_Life_Balance', 'Internet_Access', 'Political_Stability']


In [7]:
# Hybrid 1: Linear Regression on RF-selected features
# This combines nonlinear feature selection with a simple, interpretable regression model

linreg_hybrid_rf = LinearRegression()

linreg_hybrid_rf.fit(X_train_sel, y_train)
y_pred_hybrid1 = linreg_hybrid_rf.predict(X_test_sel)

print("Hybrid 1: RF Feature Selection -> Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid1)))
print("R2:", r2_score(y_test, y_pred_hybrid1))

Hybrid 1: RF Feature Selection -> Linear Regression
MAE: 1.208223044807618
RMSE: 1.4087027324699477
R2: 0.0070350019572664735


In [8]:
# Coefficients for Hybrid 1
# These coefficients help interpret which selected predictors are most strongly
# associated with life satisfaction in the final linear model

coef_df_rf = pd.DataFrame({
    "Feature": selected_features,
    "Coefficient": linreg_hybrid_rf.coef_
})

coef_df_rf["Abs_Coefficient"] = coef_df_rf["Coefficient"].abs()
coef_df_rf.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
1,Generosity,2.881764e-01,2.881764e-01
10,Political_Stability,-5.689185e-02,5.689185e-02
6,Income_Inequality,-5.584725e-03,5.584725e-03
8,Work_Life_Balance,-4.304551e-03,4.304551e-03
5,Mental_Health_Index,2.483218e-03,2.483218e-03
9,Internet_Access,-1.768923e-03,1.768923e-03
4,Urbanization_Rate,-9.267947e-04,9.267947e-04
2,Unemployment_Rate,7.190134e-04,7.190134e-04
7,Climate_Index,4.990262e-04,4.990262e-04
0,GDP_per_Capita,8.251946e-08,8.251946e-08


In [9]:
# Hybrid 2: XGBoost feature selection
# XGBoost is used as an alternative nonlinear selector
# This allows comparison with Random Forest-based feature selection

xgb_selector = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_selector.fit(X_train, y_train)

selector_xgb = SelectFromModel(
    xgb_selector,
    threshold="median"
)

selector_xgb.fit(X_train, y_train)

X_train_sel_xgb = selector_xgb.transform(X_train)
X_test_sel_xgb = selector_xgb.transform(X_test)

selected_features_xgb = X_train.columns[selector_xgb.get_support()]

print("Number of selected features:", len(selected_features_xgb))
print("\nSelected features:")
print(selected_features_xgb.tolist())

Number of selected features: 11

Selected features:
['GDP_per_Capita', 'Freedom', 'Corruption_Perception', 'Population', 'Urbanization_Rate', 'Public_Trust', 'Income_Inequality', 'Work_Life_Balance', 'Internet_Access', 'Crime_Rate', 'Employment_Rate']


In [10]:
# Hybrid 2: Linear Regression on XGBoost-selected features
# This tests whether XGBoost-based feature selection produces better final
# linear predictions than Random Forest-based feature selection

linreg_hybrid_xgb = LinearRegression()

linreg_hybrid_xgb.fit(X_train_sel_xgb, y_train)
y_pred_hybrid2 = linreg_hybrid_xgb.predict(X_test_sel_xgb)

print("Hybrid 2: XGB Feature Selection -> Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid2)))
print("R2:", r2_score(y_test, y_pred_hybrid2))

Hybrid 2: XGB Feature Selection -> Linear Regression
MAE: 1.2134286340631824
RMSE: 1.413553877258053
R2: 0.0001842859953302689


In [11]:
# Coefficients for Hybrid 2
# Again, coefficients are inspected after feature selection to interpret
# which variables remain important in the final linear model

coef_df_xgb = pd.DataFrame({
    "Feature": selected_features_xgb,
    "Coefficient": linreg_hybrid_xgb.coef_
})

coef_df_xgb["Abs_Coefficient"] = coef_df_xgb["Coefficient"].abs()
coef_df_xgb.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
1,Freedom,4.590045e-02,4.590045e-02
2,Corruption_Perception,2.451897e-02,2.451897e-02
6,Income_Inequality,-5.546171e-03,5.546171e-03
5,Public_Trust,-4.923057e-03,4.923057e-03
7,Work_Life_Balance,-4.339613e-03,4.339613e-03
10,Employment_Rate,-1.884719e-03,1.884719e-03
8,Internet_Access,-1.829898e-03,1.829898e-03
9,Crime_Rate,-1.491326e-03,1.491326e-03
4,Urbanization_Rate,-8.641266e-04,8.641266e-04
0,GDP_per_Capita,3.609847e-08,3.609847e-08


In [12]:
# Hybrid 3: Residual hybrid model
# fit a Linear Regression model, compute residuals on the training set, fit XGBoost on those residuals
# Final prediction = linear prediction + predicted residual correction

linreg_base = LinearRegression()
linreg_base.fit(X_train, y_train)

# Residuals from the linear model on training data
train_pred_lin = linreg_base.predict(X_train)
train_residuals = y_train - train_pred_lin

# XGBoost model on residuals
xgb_residual = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_residual.fit(X_train, train_residuals)

# Final predictions on test set
y_pred_lin_test = linreg_base.predict(X_test)
y_pred_residual_test = xgb_residual.predict(X_test)
y_pred_hybrid3 = y_pred_lin_test + y_pred_residual_test

print("Hybrid 3: Linear Regression + Residual XGBoost")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid3))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid3)))
print("R2:", r2_score(y_test, y_pred_hybrid3))

Hybrid 3: Linear Regression + Residual XGBoost
MAE: 1.2190150057645297
RMSE: 1.421863599026606
R2: -0.011605304501236757


In [13]:
# Inspect baseline linear coefficients from the residual hybrid
# These remain useful for interpretation because the hybrid starts with a
# transparent linear structure before applying nonlinear residual correction

coef_df_residual_base = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": linreg_base.coef_
})

coef_df_residual_base["Abs_Coefficient"] = coef_df_residual_base["Coefficient"].abs()
coef_df_residual_base.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
8,Education_Index,0.370572,0.370572
5,Generosity,0.284434,0.284434
2,Social_Support,-0.071448,0.071448
19,Political_Stability,-0.062575,0.062575
4,Freedom,0.045513,0.045513
6,Corruption_Perception,0.030690,0.030690
14,Public_Health_Expenditure,-0.009107,0.009107
13,Income_Inequality,-0.005375,0.005375
0,Year,0.004205,0.004205
12,Mental_Health_Index,0.002495,0.002495


In [14]:
# Hybrid 4: Stacking Regressor
# This combines the predictions of:
# - Linear Regression
# - Random Forest Regressor
# - XGBoost Regressor
# The final estimator is again Linear Regression

base_estimators = [
    ("lr", LinearRegression()),
    ("rf", RandomForestRegressor(
        n_estimators=100,
        max_depth=6,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

stack_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=LinearRegression(),
    cv=5,
    n_jobs=-1
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("Hybrid 4: Stacking Regressor")
print("MAE:", mean_absolute_error(y_test, y_pred_stack))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_stack)))
print("R2:", r2_score(y_test, y_pred_stack))

Hybrid 4: Stacking Regressor
MAE: 1.211277692036038
RMSE: 1.4121747992660403
R2: 0.00213419575387519


In [15]:
# Compare all hybrid models on the test set
# - Lower MAE and RMSE are better
# - Higher R2 is better

results = pd.DataFrame([
    {
        "Model": "RF selection -> Linear Regression",
        "MAE": mean_absolute_error(y_test, y_pred_hybrid1),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_hybrid1)),
        "R2": r2_score(y_test, y_pred_hybrid1),
    },
    {
        "Model": "XGB selection -> Linear Regression",
        "MAE": mean_absolute_error(y_test, y_pred_hybrid2),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_hybrid2)),
        "R2": r2_score(y_test, y_pred_hybrid2),
    },
    {
        "Model": "Linear Regression + Residual XGBoost",
        "MAE": mean_absolute_error(y_test, y_pred_hybrid3),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_hybrid3)),
        "R2": r2_score(y_test, y_pred_hybrid3),
    },
    {
        "Model": "Stacking Regressor",
        "MAE": mean_absolute_error(y_test, y_pred_stack),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_stack)),
        "R2": r2_score(y_test, y_pred_stack),
    }
])

results.sort_values("R2", ascending=False)

,Model,MAE,RMSE,R2
0,RF selection -> Linear Regression,1.208223,1.408703,0.007035
3,Stacking Regressor,1.211278,1.412175,0.002134
1,XGB selection -> Linear Regression,1.213429,1.413554,0.000184
2,Linear Regression + Residual XGBoost,1.219015,1.421864,-0.011605


In [16]:
# Compare overlap between RF- and XGB-selected features
# This helps assess whether both nonlinear selectors identify a similar
# set of relevant predictors

rf_set = set(selected_features)
xgb_set = set(selected_features_xgb)

common_features = sorted(rf_set.intersection(xgb_set))
rf_only = sorted(rf_set - xgb_set)
xgb_only = sorted(xgb_set - rf_set)

print("Number of common selected features:", len(common_features))
print("\nCommon features:")
print(common_features)

print("\nFeatures selected only by RF:")
print(rf_only)

print("\nFeatures selected only by XGB:")
print(xgb_only)

Number of common selected features: 6

Common features:
['GDP_per_Capita', 'Income_Inequality', 'Internet_Access', 'Population', 'Urbanization_Rate', 'Work_Life_Balance']

Features selected only by RF:
['Climate_Index', 'Generosity', 'Mental_Health_Index', 'Political_Stability', 'Unemployment_Rate']

Features selected only by XGB:
['Corruption_Perception', 'Crime_Rate', 'Employment_Rate', 'Freedom', 'Public_Trust']


In [17]:
# Cross-validation for the main hybrid models
# Because the dataset is small, cross-validation provides a more robust estimate
# of performance than a single split alone

models_cv = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=6,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )
}

cv_summary = []

for name, model in models_cv.items():
    scores = cross_validate(
        model,
        X, y,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        }
    )

    cv_summary.append({
        "Model": name,
        "CV_MAE": -scores["test_mae"].mean(),
        "CV_RMSE": -scores["test_rmse"].mean(),
        "CV_R2": scores["test_r2"].mean()
    })

cv_results = pd.DataFrame(cv_summary)
cv_results.sort_values("CV_R2", ascending=False)

,Model,CV_MAE,CV_RMSE,CV_R2
0,Linear Regression,1.239811,1.435794,-0.003112
1,Random Forest,1.242631,1.436131,-0.003591
2,XGBoost,1.243287,1.439716,-0.008639


In [18]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import make_scorer

scoring_hybrid = {
    "r2":   "r2",
    "mae":  make_scorer(mean_absolute_error, greater_is_better=False),
    "rmse": make_scorer(
        lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred)),
        greater_is_better=False
    )
}

# Feature-selected models need transformed X
X_sel_rf  = selector.transform(X)
X_sel_xgb = selector_xgb.transform(X)

hybrid_cv_results = {}

for name, model, X_cv in [
    ("RF Selection → LinReg",   linreg_hybrid_rf,  X_sel_rf),
    ("XGB Selection → LinReg",  linreg_hybrid_xgb, X_sel_xgb),
    ("Stacking",                stack_model,        X),
]:
    scores = cross_validate(model, X_cv, y, cv=cv,
                            scoring=scoring_hybrid, n_jobs=-1)
    hybrid_cv_results[name] = {
        "R² (mean)":   round(scores["test_r2"].mean(), 3),
        "R² (std)":    round(scores["test_r2"].std(), 3),
        "MAE (mean)":  round(-scores["test_mae"].mean(), 3),
        "RMSE (mean)": round(-scores["test_rmse"].mean(), 3),
    }

cv_hybrid_df = pd.DataFrame(hybrid_cv_results).T
print(cv_hybrid_df)
print(f"\nBaseline RMSE (predict mean): {y.std():.3f}")
print("All hybrid models perform at baseline level — confirms null result.")

                        R² (mean)  R² (std)  MAE (mean)  RMSE (mean)
RF Selection → LinReg      -0.000     0.004       1.239        1.434
XGB Selection → LinReg     -0.003     0.004       1.241        1.436
Stacking                   -0.000     0.002       1.240        1.434

Baseline RMSE (predict mean): 1.434
All hybrid models perform at baseline level — confirms null result.


In [ ]:
## Hybrid Models — Results Summary (Synthetic World Happiness Dataset)

### Approach
# Four hybrid pipelines evaluated:
# 1. RF feature selection (threshold=median) → Linear Regression
# 2. XGB feature selection (threshold=median) → Linear Regression
# 3. Linear Regression + Residual XGBoost correction
# 4. Stacking (LR + RF + XGB base learners, LR meta-learner)
#
# Note: Hybrid 3 (residual correction) was added specifically for
# this dataset as an alternative to standard stacking, given the
# absence of signal makes feature selection uninformative.

### Key Findings
# All hybrid models perform at or below baseline (R² ≈ 0):
# - RF Selection → LinReg  : R² = 0.007 (best, marginally above baseline)
# - Stacking               : R² = 0.002
# - XGB Selection → LinReg : R² = 0.000
# - Residual XGBoost       : R² = -0.012
#
# CV results confirm the null finding is stable — all CV R²
# values cluster around -0.003 to -0.009, within noise of zero.

### Feature Selection Overlap
# RF and XGB selectors agree on 6 of their selected features
# (GDP_per_Capita, Income_Inequality, Internet_Access, Population,
# Urbanization_Rate, Work_Life_Balance), with 5 features unique
# to each selector. This modest overlap reflects that both
# selectors are finding marginally correlated noise rather
# than genuine signal — consistent with the null result.

### Why Hybrid Models Cannot Help Here
# Feature selection requires genuine signal to identify relevant
# predictors. With max |r| = 0.049 across all features, no
# subset of predictors can meaningfully improve over the full
# feature set. Stacking and residual correction similarly
# cannot recover signal that does not exist in the data.
# This confirms the null result is robust to modeling strategy.

### Thesis Implication
# The consistent failure of all hybrid approaches — across four
# fundamentally different pipeline designs — provides the
# strongest possible evidence that the synthetic dataset
# contains no recoverable signal. This finding complements
# the baseline null result and strengthens the data quality
# argument. It also demonstrates that model complexity is
# not a remedy for poor data quality, an important practical
# finding for RQ5 (model robustness).

In [ ]:

## Section 2: Hybrid Models — Real WHR Gallup Poll Dataset (2005–2020)

# Same hybrid pipeline applied to the real Gallup dataset.
# Target: Life_Ladder (continuous, 0–10)
# n = 1,949 country-year observations, 149 countries

In [19]:
# Load Gallup ML dataset
df_gallup = pd.read_csv("../../data/processed/world_happiness_gallup_ml.csv")

y_gallup = df_gallup["Life_Ladder"]

X_gallup = df_gallup.drop(columns=[
    "Life_Ladder",
    "Life_Ladder__missing",
    "Country_name",
], errors="ignore")

X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_gallup, y_gallup,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train_g.shape)
print("Test shape:", X_test_g.shape)
print(f"Features: {X_gallup.columns.tolist()}")

Train shape: (1559, 17)
Test shape: (390, 17)
Features: ['year', 'Log_GDP_per_capita', 'Social_support', 'Healthy_life_expectancy_at_birth', 'Freedom_to_make_life_choices', 'Generosity', 'Perceptions_of_corruption', 'Positive_affect', 'Negative_affect', 'Log_GDP_per_capita__missing', 'Social_support__missing', 'Healthy_life_expectancy_at_birth__missing', 'Freedom_to_make_life_choices__missing', 'Generosity__missing', 'Perceptions_of_corruption__missing', 'Positive_affect__missing', 'Negative_affect__missing']


In [20]:
# Hybrid 1: RF Feature Selection → Linear Regression
rf_selector_g = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_selector_g.fit(X_train_g, y_train_g)

selector_g = SelectFromModel(rf_selector_g, threshold="median")
selector_g.fit(X_train_g, y_train_g)

X_train_sel_g  = selector_g.transform(X_train_g)
X_test_sel_g   = selector_g.transform(X_test_g)
selected_features_g = X_gallup.columns[selector_g.get_support()]

print("Number of selected features:", len(selected_features_g))
print("Selected features:", selected_features_g.tolist())

Number of selected features: 9
Selected features: ['year', 'Log_GDP_per_capita', 'Social_support', 'Healthy_life_expectancy_at_birth', 'Freedom_to_make_life_choices', 'Generosity', 'Perceptions_of_corruption', 'Positive_affect', 'Negative_affect']


In [21]:
linreg_hybrid_rf_g = LinearRegression()
linreg_hybrid_rf_g.fit(X_train_sel_g, y_train_g)
y_pred_hybrid1_g = linreg_hybrid_rf_g.predict(X_test_sel_g)

print("Hybrid 1: RF Feature Selection → Linear Regression")
print("MAE:",  mean_absolute_error(y_test_g, y_pred_hybrid1_g))
print("RMSE:", np.sqrt(mean_squared_error(y_test_g, y_pred_hybrid1_g)))
print("R2:",   r2_score(y_test_g, y_pred_hybrid1_g))

# Coefficients
coef_df_rf_g = pd.DataFrame({
    "Feature":     selected_features_g,
    "Coefficient": linreg_hybrid_rf_g.coef_
})
coef_df_rf_g["Abs"] = coef_df_rf_g["Coefficient"].abs()
print(coef_df_rf_g.sort_values("Abs", ascending=False).head(15))

Hybrid 1: RF Feature Selection → Linear Regression
MAE: 0.4305620454319461
RMSE: 0.5503212379634075
R2: 0.7608395118109287
                            Feature  Coefficient       Abs
7                   Positive_affect     1.954536  1.954536
2                    Social_support     1.722472  1.722472
4      Freedom_to_make_life_choices     0.615958  0.615958
6         Perceptions_of_corruption    -0.544983  0.544983
5                        Generosity     0.397026  0.397026
1                Log_GDP_per_capita     0.344075  0.344075
8                   Negative_affect     0.091261  0.091261
3  Healthy_life_expectancy_at_birth     0.034349  0.034349
0                              year    -0.013410  0.013410


In [22]:
# Hybrid 2: XGB Feature Selection → Linear Regression
xgb_selector_g = XGBRegressor(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_selector_g.fit(X_train_g, y_train_g)

selector_xgb_g = SelectFromModel(xgb_selector_g, threshold="median")
selector_xgb_g.fit(X_train_g, y_train_g)

X_train_sel_xgb_g  = selector_xgb_g.transform(X_train_g)
X_test_sel_xgb_g   = selector_xgb_g.transform(X_test_g)
selected_features_xgb_g = X_gallup.columns[selector_xgb_g.get_support()]

print("Number of selected features:", len(selected_features_xgb_g))
print("Selected features:", selected_features_xgb_g.tolist())

Number of selected features: 9
Selected features: ['Log_GDP_per_capita', 'Social_support', 'Healthy_life_expectancy_at_birth', 'Freedom_to_make_life_choices', 'Perceptions_of_corruption', 'Positive_affect', 'Negative_affect', 'Healthy_life_expectancy_at_birth__missing', 'Perceptions_of_corruption__missing']


In [23]:
linreg_hybrid_xgb_g = LinearRegression()
linreg_hybrid_xgb_g.fit(X_train_sel_xgb_g, y_train_g)
y_pred_hybrid2_g = linreg_hybrid_xgb_g.predict(X_test_sel_xgb_g)

print("Hybrid 2: XGB Feature Selection → Linear Regression")
print("MAE:",  mean_absolute_error(y_test_g, y_pred_hybrid2_g))
print("RMSE:", np.sqrt(mean_squared_error(y_test_g, y_pred_hybrid2_g)))
print("R2:",   r2_score(y_test_g, y_pred_hybrid2_g))

# Coefficients
coef_df_xgb_g = pd.DataFrame({
    "Feature":     selected_features_xgb_g,
    "Coefficient": linreg_hybrid_xgb_g.coef_
})
coef_df_xgb_g["Abs"] = coef_df_xgb_g["Coefficient"].abs()
print(coef_df_xgb_g.sort_values("Abs", ascending=False).head(15))

Hybrid 2: XGB Feature Selection → Linear Regression
MAE: 0.4333897193580234
RMSE: 0.553236532518144
R2: 0.7582989228950928
                                     Feature  Coefficient       Abs
5                            Positive_affect     2.200237  2.200237
1                             Social_support     1.729229  1.729229
4                  Perceptions_of_corruption    -0.605528  0.605528
3               Freedom_to_make_life_choices     0.545714  0.545714
0                         Log_GDP_per_capita     0.345941  0.345941
7  Healthy_life_expectancy_at_birth__missing    -0.140953  0.140953
8         Perceptions_of_corruption__missing    -0.088387  0.088387
6                            Negative_affect    -0.053001  0.053001
2           Healthy_life_expectancy_at_birth     0.031925  0.031925


In [24]:
# Hybrid 3: Stacking Regressor
base_estimators_g = [
    ("lr", LinearRegression()),
    ("rf", RandomForestRegressor(
        n_estimators=150,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBRegressor(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

stack_model_g = StackingRegressor(
    estimators=base_estimators_g,
    final_estimator=LinearRegression(),
    cv=5,
    n_jobs=-1
)

stack_model_g.fit(X_train_g, y_train_g)
y_pred_stack_g = stack_model_g.predict(X_test_g)

print("Hybrid 3: Stacking (LR + RF + XGB)")
print("MAE:",  mean_absolute_error(y_test_g, y_pred_stack_g))
print("RMSE:", np.sqrt(mean_squared_error(y_test_g, y_pred_stack_g)))
print("R2:",   r2_score(y_test_g, y_pred_stack_g))

Hybrid 3: Stacking (LR + RF + XGB)
MAE: 0.3337441022405371
RMSE: 0.4362970254815897
R2: 0.8496784003329729


In [25]:
# Feature selection overlap
rf_set_g  = set(selected_features_g)
xgb_set_g = set(selected_features_xgb_g)

print("Common features:", sorted(rf_set_g & xgb_set_g))
print("RF only:", sorted(rf_set_g - xgb_set_g))
print("XGB only:", sorted(xgb_set_g - rf_set_g))

Common features: ['Freedom_to_make_life_choices', 'Healthy_life_expectancy_at_birth', 'Log_GDP_per_capita', 'Negative_affect', 'Perceptions_of_corruption', 'Positive_affect', 'Social_support']
RF only: ['Generosity', 'year']
XGB only: ['Healthy_life_expectancy_at_birth__missing', 'Perceptions_of_corruption__missing']


In [26]:
# CV + summary
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer

cv_g = KFold(n_splits=5, shuffle=True, random_state=42)

scoring_g = {
    "r2":   "r2",
    "mae":  make_scorer(mean_absolute_error, greater_is_better=False),
    "rmse": make_scorer(
        lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred)),
        greater_is_better=False
    )
}

X_sel_rf_full_g  = selector_g.transform(X_gallup)
X_sel_xgb_full_g = selector_xgb_g.transform(X_gallup)

hybrid_cv_g = {}
for name, model, X_cv in [
    ("RF Selection → LinReg",    linreg_hybrid_rf_g,  X_sel_rf_full_g),
    ("XGB Selection → LinReg",   linreg_hybrid_xgb_g, X_sel_xgb_full_g),
    ("Stacking (LR + RF + XGB)", stack_model_g,        X_gallup),
]:
    scores = cross_validate(model, X_cv, y_gallup,
                            cv=cv_g, scoring=scoring_g, n_jobs=-1)
    hybrid_cv_g[name] = {
        "R² (mean)":   round(scores["test_r2"].mean(), 3),
        "R² (std)":    round(scores["test_r2"].std(), 3),
        "MAE (mean)":  round(-scores["test_mae"].mean(), 3),
        "MAE (std)":   round(scores["test_mae"].std(), 3),
        "RMSE (mean)": round(-scores["test_rmse"].mean(), 3),
        "RMSE (std)":  round(scores["test_rmse"].std(), 3),
    }

cv_hybrid_df_g = pd.DataFrame(hybrid_cv_g).T
print(cv_hybrid_df_g)

                          R² (mean)  R² (std)  MAE (mean)  MAE (std)  \
RF Selection → LinReg         0.759     0.007       0.425      0.013   
XGB Selection → LinReg        0.755     0.008       0.427      0.013   
Stacking (LR + RF + XGB)      0.840     0.007       0.344      0.012   

                          RMSE (mean)  RMSE (std)  
RF Selection → LinReg           0.547       0.018  
XGB Selection → LinReg          0.551       0.019  
Stacking (LR + RF + XGB)        0.446       0.018  


In [27]:
# Summary results table
results_hybrid_g = pd.DataFrame([
    {
        "Model":        "RF Selection → LinReg",
        "MAE (test)":   round(mean_absolute_error(y_test_g, y_pred_hybrid1_g), 3),
        "RMSE (test)":  round(np.sqrt(mean_squared_error(y_test_g, y_pred_hybrid1_g)), 3),
        "R² (test)":    round(r2_score(y_test_g, y_pred_hybrid1_g), 3),
        "CV R²":        cv_hybrid_df_g.loc["RF Selection → LinReg", "R² (mean)"],
        "CV R² std":    cv_hybrid_df_g.loc["RF Selection → LinReg", "R² (std)"],
        "CV MAE":       cv_hybrid_df_g.loc["RF Selection → LinReg", "MAE (mean)"],
        "CV RMSE":      cv_hybrid_df_g.loc["RF Selection → LinReg", "RMSE (mean)"],
    },
    {
        "Model":        "XGB Selection → LinReg",
        "MAE (test)":   round(mean_absolute_error(y_test_g, y_pred_hybrid2_g), 3),
        "RMSE (test)":  round(np.sqrt(mean_squared_error(y_test_g, y_pred_hybrid2_g)), 3),
        "R² (test)":    round(r2_score(y_test_g, y_pred_hybrid2_g), 3),
        "CV R²":        cv_hybrid_df_g.loc["XGB Selection → LinReg", "R² (mean)"],
        "CV R² std":    cv_hybrid_df_g.loc["XGB Selection → LinReg", "R² (std)"],
        "CV MAE":       cv_hybrid_df_g.loc["XGB Selection → LinReg", "MAE (mean)"],
        "CV RMSE":      cv_hybrid_df_g.loc["XGB Selection → LinReg", "RMSE (mean)"],
    },
    {
        "Model":        "Stacking (LR + RF + XGB)",
        "MAE (test)":   round(mean_absolute_error(y_test_g, y_pred_stack_g), 3),
        "RMSE (test)":  round(np.sqrt(mean_squared_error(y_test_g, y_pred_stack_g)), 3),
        "R² (test)":    round(r2_score(y_test_g, y_pred_stack_g), 3),
        "CV R²":        cv_hybrid_df_g.loc["Stacking (LR + RF + XGB)", "R² (mean)"],
        "CV R² std":    cv_hybrid_df_g.loc["Stacking (LR + RF + XGB)", "R² (std)"],
        "CV MAE":       cv_hybrid_df_g.loc["Stacking (LR + RF + XGB)", "MAE (mean)"],
        "CV RMSE":      cv_hybrid_df_g.loc["Stacking (LR + RF + XGB)", "RMSE (mean)"],
    },
]).set_index("Model")

print(results_hybrid_g)

                          MAE (test)  RMSE (test)  R² (test)  CV R²  \
Model                                                                 
RF Selection → LinReg          0.431        0.550      0.761  0.759   
XGB Selection → LinReg         0.433        0.553      0.758  0.755   
Stacking (LR + RF + XGB)       0.334        0.436      0.850  0.840   

                          CV R² std  CV MAE  CV RMSE  
Model                                                 
RF Selection → LinReg         0.007   0.425    0.547  
XGB Selection → LinReg        0.008   0.427    0.551  
Stacking (LR + RF + XGB)      0.007   0.344    0.446  


In [ ]:
# ## Hybrid Models — Results Summary (World Happiness Report Notebooks)
#
# ---
#
# ### Section 1: Synthetic Kaggle Dataset
#
# All hybrid models perform at baseline level (R² ≈ 0), consistent
# with the baseline notebook findings. Feature selection, residual
# correction, and stacking all fail to recover signal that does not
# exist in the data. The null result is robust to modeling strategy.
#
# Best hybrid: RF Selection → LinReg (R² = 0.007, effectively zero)
# Baseline RMSE: 1.434 — all hybrid RMSEs within 0.01 of baseline.
#
# ---
#
# ### Section 2: Real WHR Gallup Poll Dataset
#
# ### Approach
# Three hybrid pipelines evaluated:
# 1. RF feature selection (threshold=median, 9 features) → Linear Regression
# 2. XGB feature selection (threshold=median, 9 features) → Linear Regression
# 3. Stacking (LR + RF + XGB base learners, LR meta-learner, cv=5)
#
# ### Key Findings
# - Stacking is the best hybrid model (CV R²: 0.840, CV RMSE: 0.446),
#   matching the best baseline (XGBoost, CV R²: 0.850) within margin.
# - RF and XGB selection pipelines perform nearly identically
#   (CV R²: 0.759 vs 0.755) and match the baseline Linear Regression
#   (CV R²: 0.763) — feature selection adds no value here.
# - CV std is very low across all models (0.007–0.008), confirming
#   stable generalization.
# - Test set and CV metrics are nearly identical, confirming no
#   overfitting to the test split.
#
# ### Feature Selection
# Both RF and XGB selectors agree on 7 core features:
# Freedom_to_make_life_choices, Healthy_life_expectancy,
# Log_GDP_per_capita, Negative_affect, Perceptions_of_corruption,
# Positive_affect, Social_support. This is the entire substantive
# feature set — both selectors discard only Generosity, year, and
# missingness indicators. The high overlap (7/9 features shared)
# confirms the signal is concentrated in a stable set of predictors.
#
# ### Coefficient Interpretation (Selection Pipelines)
# Positive_affect (coef ~2.0–2.2) and Social_support (~1.7) are
# the strongest positive predictors in the final linear models,
# followed by Freedom_to_make_life_choices (~0.6) and GDP (~0.34).
# Perceptions_of_corruption is the strongest negative predictor
# (coef ~-0.54 to -0.61). This ordering is consistent with the
# permutation importance rankings from the baseline notebook and
# aligns with established WHR findings.
#
# ### Comparison with Baseline Models
# Hybrid models do not outperform the best baselines:
#
# | Model | CV R² | CV RMSE |
# |---|---|---|
# | XGBoost (baseline) | 0.850 | 0.432 |
# | Random Forest (baseline) | 0.843 | 0.442 |
# | Stacking (hybrid) | 0.840 | 0.446 |
# | RF Selection → LinReg | 0.759 | 0.547 |
# | XGB Selection → LinReg | 0.755 | 0.551 |
# | Linear Regression (baseline) | 0.763 | 0.543 |
#
# Stacking matches but does not exceed the best baseline.
# Feature selection pipelines match but do not exceed Linear
# Regression alone — reducing to 9 features loses nothing but
# gains nothing either, consistent with findings across all
# other datasets in this study.
#
# ### Cross-Dataset Pattern — Hybrid Models
# Across all five datasets, hybrid models consistently fail to
# outperform well-configured baseline models:
# - MH Tech: best hybrid CV F1 = 0.488 vs best baseline 0.470
# - Student Depression: hybrid = baseline (0.840 vs 0.841)
# - Canadian Survey: stacking = XGBoost (0.423 vs 0.422)
# - WHR Gallup: stacking ≈ XGBoost (0.840 vs 0.850)
# - WHR Synthetic: all models fail equally (R² ≈ 0)
#
# This is a consistent and thesis-worthy finding — for wellbeing
# prediction tasks, the signal structure is stable enough that
# simple well-configured baselines capture it fully. Hybrid
# complexity adds implementation cost without predictive gain,
# directly addressing the comparative modeling question in RQ1.